# HabitGuard 예측 모델 Colab

이 노트북은 Android 앱과 분리해서 **예측 모델 학습, 평가, 시각화, Android 추론 bundle export**만 실행합니다.

- 회귀 모델: 다음 날 총 스크린타임 예측
- 분류 모델: 다음 날 개인 목표 초과 위험 분류
- 선택 모델: `linear_regression`, `logistic_regression`
- Android export: `android_inference_bundle.json`

> 주의: CSV를 업로드하지 않으면 deterministic synthetic sample로 실행됩니다. 이 경우 결과는 `source_type=synthetic`, `evaluation_scope=synthetic evaluation`이며 실제 사용자 성능으로 주장하면 안 됩니다.

## 1. 환경 준비

GitHub 저장소를 Colab 런타임에 클론하고, 기존 프로젝트의 `ai/train_from_phone_csv.py`를 그대로 사용합니다.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

from pathlib import Path
import json
import os
import shutil
import subprocess
import textwrap

REPO_URL = "https://github.com/sionkk1/data.git"
REPO_DIR = Path("/content/data")
PROJECT_DIR = REPO_DIR / "habitguard_android"

if not (PROJECT_DIR / "ai" / "train_from_phone_csv.py").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("project:", PROJECT_DIR)
print("pipeline:", PROJECT_DIR / "ai" / "train_from_phone_csv.py")

## 2. 데이터 선택

- 실제 앱에서 내보낸 CSV가 있으면 `USE_UPLOAD=True`로 바꾸고 업로드합니다.
- CSV를 업로드하지 않으면 합성 샘플 데이터로 파이프라인 구조만 검증합니다.
- 합성 데이터와 실제 데이터를 섞으면 안 됩니다.

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}
CSV_PATH = ""  # @param {type:"string"}

if USE_UPLOAD:
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("업로드된 파일이 없습니다.")
        CSV_PATH = next(iter(uploaded.keys()))
    except ModuleNotFoundError:
        raise RuntimeError("이 셀의 업로드 기능은 Google Colab에서 실행할 때 사용할 수 있습니다.")

if CSV_PATH:
    print("입력 CSV:", CSV_PATH)
else:
    print("입력 CSV 없음: synthetic sample로 실행합니다.")

## 3. 학습 실행

기존 Python 파이프라인을 실행합니다. 이 파이프라인은 무작위 행 80:20 분할이 아니라 사용자 단위 시간순 train/validation/test 분할을 사용합니다.

In [ ]:
OUTPUT_DIR = Path("ai/colab_outputs")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

cmd = ["python", "ai/train_from_phone_csv.py"]
if CSV_PATH:
    cmd.append(CSV_PATH)
cmd += ["--output-dir", str(OUTPUT_DIR)]

print("실행 명령:", " ".join(cmd))
completed = subprocess.run(cmd, check=True, text=True, capture_output=True)
print(completed.stdout[-4000:])

## 4. 핵심 결과 확인

아래 표는 baseline과 학습 모델을 비교합니다. `source_type`과 `evaluation_scope`를 먼저 확인하세요.

In [ ]:
import pandas as pd

manifest = json.loads((OUTPUT_DIR / "training_manifest.json").read_text(encoding="utf-8"))
regression_metrics = json.loads((OUTPUT_DIR / "regression_metrics.json").read_text(encoding="utf-8"))
classification_metrics = json.loads((OUTPUT_DIR / "classification_metrics.json").read_text(encoding="utf-8"))

print("source_type:", manifest["source_type"])
print("evaluation_scope:", manifest["evaluation_scope"])
print("split_method:", manifest["split_method"])
print("data_snapshot_hash:", manifest["data_snapshot_hash"])

comparison = pd.read_csv(OUTPUT_DIR / "model_comparison.csv")
display(comparison)

In [ ]:
summary = pd.DataFrame([
    {
        "task": "regression",
        "best_model": regression_metrics["best_model"],
        "best_baseline": regression_metrics["best_baseline"],
        "MAE": regression_metrics["mae"],
        "RMSE": regression_metrics["rmse"],
        "R2": regression_metrics["r2"],
        "baseline_improvement_pct": regression_metrics["baseline_improvement_pct"],
    },
    {
        "task": "classification",
        "best_model": classification_metrics["best_model"],
        "best_baseline": classification_metrics["best_baseline"],
        "accuracy": classification_metrics["accuracy"],
        "macro_f1": classification_metrics["macro_f1"],
        "high_risk_recall": classification_metrics["high_risk_recall"],
        "baseline_improvement_pct": classification_metrics["baseline_improvement_pct"],
    },
])
display(summary)

## 5. 시각화

- 실제값 vs 예측값
- feature importance
- confusion matrix

In [ ]:
from IPython.display import Image, display

for name in ["actual_vs_predicted.png", "feature_importance.png", "confusion_matrix.png"]:
    print(name)
    display(Image(filename=str(OUTPUT_DIR / name)))

## 6. Android 로컬 추론 bundle 확인

Android 앱은 `.joblib`을 직접 읽지 않고, 아래 JSON bundle의 feature 순서, 결측값 처리, scaler, 계수, 절편을 Kotlin에서 계산합니다.

In [ ]:
bundle = json.loads((OUTPUT_DIR / "android_inference_bundle.json").read_text(encoding="utf-8"))

print("schema_version:", bundle["schema_version"])
print("model_version:", bundle["model_version"])
print("source_type:", bundle["source_type"])
print("evaluation_scope:", bundle["evaluation_scope"])
print("regression:", bundle["regression_model"]["name"])
print("classification:", bundle["classification_model"]["name"])
print("positive_class:", bundle["classification_model"]["positive_class"])
print("feature_count:", len(bundle["feature_names"]))

pd.DataFrame({"feature_order": range(len(bundle["feature_names"])), "feature_name": bundle["feature_names"]})

## 7. 산출물 검증

In [ ]:
required_files = [
    "feature_schema.json",
    "preprocessing.joblib",
    "screen_time_regressor.joblib",
    "goal_risk_classifier.joblib",
    "user_type_classifier.joblib",
    "regression_metrics.json",
    "classification_metrics.json",
    "model_card.md",
    "training_manifest.json",
    "data_snapshot_hash.txt",
    "actual_vs_predicted.png",
    "feature_importance.png",
    "confusion_matrix.png",
    "model_comparison.csv",
    "android_inference_bundle.json",
]

missing = [name for name in required_files if not (OUTPUT_DIR / name).exists()]
if missing:
    raise RuntimeError(f"누락된 산출물: {missing}")

for forbidden in ["target_next_day_minutes", "target_goal_exceeded", "goal_risk_label", "user_type_label"]:
    if forbidden in bundle["feature_names"]:
        raise RuntimeError(f"미래 정답 열이 feature에 포함됨: {forbidden}")

print("필수 산출물과 feature leakage 검증 통과")

## 8. 다운로드

`habitguard_prediction_outputs.zip` 안에 metrics, 그래프, model card, Android inference bundle이 들어갑니다.

In [ ]:
zip_base = Path("/content/habitguard_prediction_outputs")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("zip:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ModuleNotFoundError:
    print("Colab이 아니면 위 zip 경로에서 파일을 확인하세요.")